# 16 — Export Smart Shopping Assistant to Neon

## Objective

Publish the client-facing Smart Shopping Assistant recommendations
from Databricks to Neon PostgreSQL.

Source:
`workspace.ml_data.shopping_assistant_recommendations`

Destination:
`shopping_assistant_recommendations`

The exported table will later be consumed by the backend API and web application.

In [0]:
from pyspark.sql import functions as F

source_table = "workspace.ml_data.shopping_assistant_recommendations"
target_table = "shopping_assistant_recommendations"

print("Source:", source_table)
print("Destination:", target_table)

Source: workspace.ml_data.shopping_assistant_recommendations
Destination: shopping_assistant_recommendations


In [0]:
shopping_df = spark.table(source_table)

print("Rows to export:", shopping_df.count())

print(
    "Customers:",
    shopping_df
    .select("user_id")
    .distinct()
    .count()
)

display(
    shopping_df
    .orderBy("user_id", "recommendation_rank")
    .limit(10)
)

Rows to export: 131800
Customers: 26360


user_id,target_order_id,product_id,recommendation_rank,product_name,aisle,department,purchase_probability,recommendation_type,reorder_status,shopping_section,client_action,candidate_source,why_recommended,orders_since_last_product_purchase,user_product_avg_order_gap,user_product_due_score,user_product_reorder_rate,customer_aisle_affinity,customer_department_affinity
14,2316178,29509,1,80 Vodka Holiday Edition,spirits,alcohol,0.6871980336715175,REORDER,DUE_NOW,REORDER_NOW,Add again,reorder,This product is around your usual reorder time: every order.,1,1.0,1.0,0.9231,0.0619,0.0619
14,2316178,23803,2,Jalapeno Pepper,fresh vegetables,produce,0.6460582185396573,REORDER,DUE_NOW,REORDER_NOW,Add again,reorder,This product is around your usual reorder time: every order.,1,1.0,1.0,0.9167,0.1095,0.1524
14,2316178,8744,3,Mixed Vegetables,frozen produce,frozen,0.5080867030927294,REORDER,DUE_SOON,COMING_UP,Remind me later,reorder,You are approaching your usual reorder time: every 1.1 orders.,1,1.14,0.88,0.875,0.0762,0.1667
14,2316178,37266,4,Tater Treats Seasoned Shredded Potatoes,frozen appetizers sides,frozen,0.36486852088734056,REORDER,EARLY,FAVORITES_FOR_LATER,Save for later,reorder,"A frequent favorite, but it is earlier than your usual reorder time. Historical reorder rate: 80%.",1,1.5,0.67,0.8,0.0619,0.1667
14,2316178,15869,5,Sweet Hot Dog Buns,buns rolls,bakery,0.293682771356237,REORDER,DUE_NOW,REORDER_NOW,Add again,reorder,This product is around your usual reorder time: every order.,1,1.0,1.0,0.6667,0.0143,0.0667
21,1854765,23729,1,Hard Boiled Eggs,eggs,dairy eggs,0.5683535426858906,REORDER,OVERDUE,REORDER_NOW,Add again,reorder,"You usually buy this product every 1.6 orders, and it has been 2 orders since your last purchase.",2,1.55,1.29,0.9524,0.1268,0.2488
21,1854765,44632,2,Sparkling Water Grapefruit,water seltzer sparkling water,beverages,0.3617915822651947,REORDER,EARLY,FAVORITES_FOR_LATER,Save for later,reorder,"A frequent favorite, but it is earlier than your usual reorder time. Historical reorder rate: 83%.",2,3.2,0.63,0.8333,0.0585,0.2439
21,1854765,28204,3,Organic Fuji Apple,fresh fruits,produce,0.33565992748081386,REORDER,EARLY,FAVORITES_FOR_LATER,Save for later,reorder,"A frequent favorite, but it is earlier than your usual reorder time. Historical reorder rate: 83%.",1,4.4,0.23,0.8333,0.1073,0.1463
21,1854765,48988,4,Unsweetened Premium Iced Tea,tea,beverages,0.27522307917062994,REORDER,OVERDUE,REORDER_NOW,Add again,reorder,"You usually buy this product every 1.7 orders, and it has been 3 orders since your last purchase.",3,1.65,1.82,0.9444,0.1171,0.2439
21,1854765,33894,5,Goldfish Cheddar Baked Snack Crackers Multi Packs,crackers,snacks,0.24767552364038758,REORDER,OVERDUE,REORDER_NOW,Add again,reorder,"You usually buy this product every 1.5 orders, and it has been 2 orders since your last purchase.",2,1.5,1.33,0.6667,0.0585,0.1951


In [0]:
required_columns = [
    "user_id",
    "target_order_id",
    "product_id",
    "recommendation_rank",
    "product_name",
    "purchase_probability",
    "recommendation_type",
    "reorder_status",
    "shopping_section",
    "client_action",
    "why_recommended"
]

missing_columns = [
    col
    for col in required_columns
    if col not in shopping_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


row_count = shopping_df.count()

if row_count == 0:
    raise ValueError(
        "Shopping Assistant export is empty."
    )


duplicate_count = (
    shopping_df
    .groupBy(
        "user_id",
        "target_order_id",
        "product_id"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_count > 0:
    raise ValueError(
        f"Found {duplicate_count} duplicate recommendation rows."
    )


invalid_rank_count = (
    shopping_df
    .filter(
        (F.col("recommendation_rank") < 1) |
        (F.col("recommendation_rank") > 5)
    )
    .count()
)

if invalid_rank_count > 0:
    raise ValueError(
        f"Found {invalid_rank_count} invalid recommendation ranks."
    )


print("Pre-export validation passed.")
print("Rows:", row_count)
print("Duplicates:", duplicate_count)
print("Invalid ranks:", invalid_rank_count)

Pre-export validation passed.
Rows: 131800
Duplicates: 0
Invalid ranks: 0


In [0]:
# ============================================================
# 5. Neon PostgreSQL connection
# Reuse the same configuration as the existing serving pipeline
# ============================================================

neon_host = (
    "ep-summer-sea-ag9wnzfy-pooler."
    "c-2.eu-central-1.aws.neon.tech"
)

neon_port = "5432"
neon_database = "neondb"
neon_user = "neondb_owner"

# Only the password is stored as a Databricks secret
neon_password = dbutils.secrets.get(
    scope="neon",
    key="password"
)

print("Neon configuration loaded successfully.")
print("Host:", neon_host)
print("Database:", neon_database)
print("User:", neon_user)

Neon configuration loaded successfully.
Host: ep-summer-sea-ag9wnzfy-pooler.c-2.eu-central-1.aws.neon.tech
Database: neondb
User: neondb_owner


In [0]:
# ============================================================
# 6. Export Smart Shopping Assistant recommendations to Neon
# ============================================================

target_table = "shopping_assistant_recommendations"

(
    shopping_df
    .write
    .format("postgresql")
    .option("host", neon_host)
    .option("port", neon_port)
    .option("database", neon_database)
    .option("dbtable", target_table)
    .option("user", neon_user)
    .option("password", neon_password)
    .option("batchsize", "5000")
    .option("numPartitions", "2")
    .mode("overwrite")
    .save()
)

print("Export completed:", target_table)
print("Rows exported:", shopping_df.count())

Export completed: shopping_assistant_recommendations
Rows exported: 131800


In [0]:
# ============================================================
# 7. Verify data stored in Neon PostgreSQL
# ============================================================

neon_shopping_df = (
    spark.read
    .format("postgresql")
    .option("host", neon_host)
    .option("port", neon_port)
    .option("database", neon_database)
    .option("dbtable", target_table)
    .option("user", neon_user)
    .option("password", neon_password)
    .load()
)

neon_row_count = neon_shopping_df.count()

neon_customer_count = (
    neon_shopping_df
    .select("user_id")
    .distinct()
    .count()
)

print("Neon rows:", neon_row_count)
print("Neon customers:", neon_customer_count)

Neon rows: 131800
Neon customers: 26360


In [0]:
# ============================================================
# 8. Post-export integrity validation
# ============================================================

source_count = shopping_df.count()
destination_count = neon_shopping_df.count()

if source_count != destination_count:
    raise ValueError(
        f"Row count mismatch: Databricks={source_count}, "
        f"Neon={destination_count}"
    )

duplicate_count = (
    neon_shopping_df
    .groupBy(
        "user_id",
        "target_order_id",
        "product_id"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_count > 0:
    raise ValueError(
        f"Neon contains {duplicate_count} duplicate recommendations."
    )

print("Post-export validation passed.")
print("Databricks rows:", source_count)
print("Neon rows:", destination_count)
print("Duplicate rows:", duplicate_count)

Post-export validation passed.
Databricks rows: 131800
Neon rows: 131800
Duplicate rows: 0


In [0]:
display(
    neon_shopping_df
    .select(
        "user_id",
        "recommendation_rank",
        "product_name",
        "purchase_probability",
        "reorder_status",
        "shopping_section",
        "client_action",
        "why_recommended"
    )
    .orderBy(
        "user_id",
        "recommendation_rank"
    )
    .limit(20)
)

user_id,recommendation_rank,product_name,purchase_probability,reorder_status,shopping_section,client_action,why_recommended
14,1,80 Vodka Holiday Edition,0.6871980336715175,DUE_NOW,REORDER_NOW,Add again,This product is around your usual reorder time: every order.
14,2,Jalapeno Pepper,0.6460582185396573,DUE_NOW,REORDER_NOW,Add again,This product is around your usual reorder time: every order.
14,3,Mixed Vegetables,0.5080867030927294,DUE_SOON,COMING_UP,Remind me later,You are approaching your usual reorder time: every 1.1 orders.
14,4,Tater Treats Seasoned Shredded Potatoes,0.36486852088734056,EARLY,FAVORITES_FOR_LATER,Save for later,"A frequent favorite, but it is earlier than your usual reorder time. Historical reorder rate: 80%."
14,5,Sweet Hot Dog Buns,0.293682771356237,DUE_NOW,REORDER_NOW,Add again,This product is around your usual reorder time: every order.
21,1,Hard Boiled Eggs,0.5683535426858906,OVERDUE,REORDER_NOW,Add again,"You usually buy this product every 1.6 orders, and it has been 2 orders since your last purchase."
21,2,Sparkling Water Grapefruit,0.3617915822651947,EARLY,FAVORITES_FOR_LATER,Save for later,"A frequent favorite, but it is earlier than your usual reorder time. Historical reorder rate: 83%."
21,3,Organic Fuji Apple,0.33565992748081386,EARLY,FAVORITES_FOR_LATER,Save for later,"A frequent favorite, but it is earlier than your usual reorder time. Historical reorder rate: 83%."
21,4,Unsweetened Premium Iced Tea,0.27522307917062994,OVERDUE,REORDER_NOW,Add again,"You usually buy this product every 1.7 orders, and it has been 3 orders since your last purchase."
21,5,Goldfish Cheddar Baked Snack Crackers Multi Packs,0.24767552364038758,OVERDUE,REORDER_NOW,Add again,"You usually buy this product every 1.5 orders, and it has been 2 orders since your last purchase."
